In [ ]:
# Setup input parameters
from datetime import datetime as dt
dbutils.widgets.text('bg_loadtimestamp', '')
bg_loadtimestamp = dbutils.widgets.get('bg_loadtimestamp')
bg_loadtimestamp_dt = dt.strptime(bg_loadtimestamp, '%Y-%m-%d %H:%M:%S.%f')
if not bg_loadtimestamp:
    bg_loadtimestamp = 'CAST(NULL AS Timestamp)'
else:
    bg_loadtimestamp = f"CAST('{bg_loadtimestamp}' AS Timestamp)"




In [ ]:
# Setup logging
def info(statement_name, target_object_database_name, target_object_schema_name, target_object_name, message):
    log_message('INFO', statement_name, target_object_database_name, target_object_schema_name, target_object_name, message)

def error(statement_name, target_object_database_name, target_object_schema_name, target_object_name, message):
    log_message('ERROR',statement_name, target_object_database_name, target_object_schema_name, target_object_name, message)

def log_message(log_level, statement_name, target_object_database_name, target_object_schema_name, target_object_name, message):
    log_df = spark.createDataFrame([(dt.now(), log_level, 'loader', 'MB_Databricks_Stage_JDBCGenerator', '{loadcontrol#loadcontrol#application_name}', '{loadcontrol#loadcontrol#application_environment_name}', bg_loadtimestamp_dt, statement_name, 'STG_ST_orders_Loader', target_object_database_name, target_object_schema_name, target_object_name,  message)], ['log_timestamp', 'log_level', 'execution_unit', 'project_name', 'application_name', 'application_environment_name', 'load_timestamp', 'statement_name', 'task_name', 'target_object_database_name', 'target_object_schema_name', 'target_object_name', 'message'])
    log_df.write.format('delta').mode('append').saveAsTable('`{loadcontrol#loadcontrol#database_name}`.`{loadcontrol#loadcontrol#schema_name}`.`{loadcontrol#loadcontrol#log_table_name}`')
    print(f"{dt.now().strftime('%Y/%m/%d, %H:%M:%S')} - {target_object_name}: {message}")




In [ ]:
# StageLoader: orders_Stage Loader_1

try:

    from pyspark.sql.functions import input_file_name

    from pyspark.sql.functions import col

    df = spark.read.jdbc('{mb_databricks_stage_jdbcgenerator#bigenius-source-server.database.windows.net#jdbc_url}', 'dbo.orders', properties={'user': {mb_databricks_stage_jdbcgenerator#bigenius-source-server.database.windows.net#jdbc_user}, 'password': {mb_databricks_stage_jdbcgenerator#bigenius-source-server.database.windows.net#jdbc_password}, 'driver':'{mb_databricks_stage_jdbcgenerator#bigenius-source-server.database.windows.net#jdbc_driver}'})
    df.createOrReplaceTempView("`stg_st_orders_source_temp`")

    spark.sql(f"""CREATE OR REPLACE TEMPORARY VIEW `stg_st_orders_source`
    AS
    SELECT
         CAST(NULL AS Timestamp) AS `bg_loadtimestamp`
        ,CAST(NULL AS String) AS `bg_sourcesystem`
        ,`s1`.`order_id` AS `order_id`
        ,`s1`.`order_tms` AS `order_tms`
        ,`s1`.`customer_id` AS `customer_id`
        ,`s1`.`order_status` AS `order_status`
        ,`s1`.`store_id` AS `store_id`
    FROM `stg_st_orders_source_temp` AS `s1`
    """)

    operation_metrics_collection = {}
    result_df = spark.sql(f"""
    INSERT
    OVERWRITE `{mb_databricks_stage_jdbcgenerator#stage#database_name}`.`{mb_databricks_stage_jdbcgenerator#stage#schema_name}`.`stg_st_orders` (
         `bg_loadtimestamp`
        ,`bg_sourcesystem`
        ,`order_id`
        ,`order_tms`
        ,`customer_id`
        ,`order_status`
        ,`store_id`
    )
    SELECT
         {bg_loadtimestamp} AS `bg_loadtimestamp`
        ,`bg_source`.`bg_sourcesystem` AS `bg_sourcesystem`
        ,`bg_source`.`order_id` AS `order_id`
        ,`bg_source`.`order_tms` AS `order_tms`
        ,`bg_source`.`customer_id` AS `customer_id`
        ,`bg_source`.`order_status` AS `order_status`
        ,`bg_source`.`store_id` AS `store_id`
    FROM `stg_st_orders_source` AS `bg_source`
    """)
    RowCountInserted = result_df.select("num_inserted_rows").collect()[0][0]
    pandas_df = result_df.toPandas()
    operation_metrics = pandas_df.to_dict(orient='records')
    operation_metrics_collection['reloadtarget_{mb_databricks_stage_jdbcgenerator#stage#database_name}_{mb_databricks_stage_jdbcgenerator#stage#schema_name}_stg_st_orders'] = operation_metrics
    info('ReloadTarget', '{mb_databricks_stage_jdbcgenerator#stage#database_name}', '{mb_databricks_stage_jdbcgenerator#stage#schema_name}', 'STG_ST_orders', str(operation_metrics))

except Exception as e:
    error('', '', '', '', str(e))
    raise



In [ ]:
dbutils.notebook.exit(operation_metrics_collection)
